# Topic 3: AWS EMR Deep Dive

> **Phase 7 → Week 13 → Topic 3**

---

## What This Covers

Week 12 covered EMR at an overview level. This notebook goes deep on EMR cluster design, cost patterns, bootstrap actions, Step Functions integration, and EMR Serverless.

---

## Interview Cheat Sheet

**Q: What are Spot instances in AWS EMR and what are the risks?**
> Spot instances are unused EC2 capacity sold at 60-80% discount. AWS can reclaim them with a 2-minute warning. For EMR: use On-Demand for the driver/master node (interruption kills the job), use Spot for TASK instance groups (purely compute, no HDFS data). EMR automatically handles task node reclamation by redistributing in-flight tasks. Risk: if too many Spot nodes are reclaimed simultaneously, job slows or fails.

**Q: What is EMR Serverless and how does it differ from a regular EMR cluster?**
> EMR Serverless is a fully serverless Spark/Hive runtime — you submit a job and AWS provisions, runs, and tears down capacity automatically. No cluster to manage, no idle cost, pay per vCPU-second + GB-second of memory used. Regular EMR: you manage cluster size, lifecycle, and pay for idle time. EMR Serverless: best for sporadic jobs or teams without cluster management expertise. Tradeoff: cold start (~30-60 seconds) and less fine-grained tuning control.

**Q: What are EMR Bootstrap Actions?**
> Bootstrap actions are shell scripts that run on every EMR node before Spark starts. Use them to: install Python packages (pip install), copy config files from S3, install system packages (yum install), set environment variables, configure JVM options. Bootstrap actions run as root, so they can modify any system config.

In [ ]:
print("AWS EMR Deep Dive — reference patterns")
print("Code requires AWS credentials and infrastructure to run.")

## Part 1: Cluster Architecture & Instance Groups

In [ ]:
print("""
EMR Cluster Architecture:
══════════════════════════════════════════════════════════════

Instance Groups (3 types):
┌─────────────────────────────────────────────────────────────┐
│  MASTER   (1 node)                                          │
│  - YARN ResourceManager                                     │
│  - HDFS NameNode                                            │
│  - Spark Driver (when --deploy-mode cluster)                │
│  ⚠ ALWAYS On-Demand — Spot here kills the entire job       │
├─────────────────────────────────────────────────────────────┤
│  CORE     (1+ nodes)                                        │
│  - YARN NodeManager                                         │
│  - HDFS DataNode (stores data)                              │
│  ⚠ On-Demand recommended — losing a CORE node = HDFS data  │
│    loss (use replication factor 3 if using HDFS)           │
│  ✓ If reading from S3 only (no HDFS), Spot is OK            │
├─────────────────────────────────────────────────────────────┤
│  TASK     (0+ nodes, optional)                              │
│  - YARN NodeManager only (pure compute, no HDFS)            │
│  - Can be added/removed at any time                         │
│  ✓ ALWAYS use Spot here — safe to lose, AWS redistributes  │
└─────────────────────────────────────────────────────────────┘

Best practice for S3-based workloads (most common):
  MASTER:  1 × m5.xlarge   (On-Demand)
  CORE:    2 × m5.xlarge   (On-Demand, minimum for HA)
  TASK:    N × m5.2xlarge  (Spot, 60-80% savings)

aws emr create-cluster example:
  aws emr create-cluster \\
    --name "Production ETL" \\
    --release-label emr-6.15.0 \\
    --applications Name=Spark Name=Hive \\
    --instance-groups \\
      InstanceGroupType=MASTER,InstanceCount=1,InstanceType=m5.xlarge \\
      InstanceGroupType=CORE,InstanceCount=2,InstanceType=m5.2xlarge \\
      InstanceGroupType=TASK,InstanceCount=8,InstanceType=m5.2xlarge,BidPrice=OnDemandPrice \\
    --use-default-roles \\
    --auto-terminate \\
    --log-uri s3://my-logs-bucket/emr/ \\
    --bootstrap-actions Path=s3://my-bucket/bootstrap.sh,Args=[]
══════════════════════════════════════════════════════════════
""")

## Part 2: Bootstrap Actions

In [ ]:
print("""
EMR Bootstrap Actions — bootstrap.sh:
══════════════════════════════════════════════════════════════

#!/bin/bash
set -ex

# 1. Install Python packages on all nodes
pip3 install \\
    great-expectations==0.18.0 \\
    boto3==1.34.0 \\
    delta-spark==3.0.0

# 2. Copy config files from S3
aws s3 cp s3://my-bucket/config/log4j.properties /etc/spark/conf/log4j.properties
aws s3 cp s3://my-bucket/config/spark-defaults.conf /etc/spark/conf/spark-defaults.conf

# 3. Set environment variables
echo 'export PYSPARK_PYTHON=/usr/bin/python3' >> /etc/environment
echo 'export JAVA_HOME=/usr/lib/jvm/java-17' >> /etc/environment

# 4. Install system packages
sudo yum install -y jq curl

# 5. Conditionally run only on master
IS_MASTER=$(cat /mnt/var/lib/info/instance.json | jq -r .isMaster)
if [ "$IS_MASTER" = "true" ]; then
    # Only runs on master node
    pip3 install airflow-providers-amazon
    echo "Master-only setup complete"
fi

echo "Bootstrap complete"

Key bootstrap tips:
  - set -ex: exit on any error (don't hide failures)
  - Bootstrap failures block cluster start (EMR waits for all nodes)
  - Keep bootstrap fast — every extra minute = cost for every node
  - Bake frequently-used packages into a custom AMI instead
══════════════════════════════════════════════════════════════
""")

## Part 3: Spark Configuration on EMR

In [ ]:
print("""
EMR Spark Configuration — spark-defaults.conf:
══════════════════════════════════════════════════════════════

Pass configurations at cluster creation:
  aws emr create-cluster \\
    --configurations '[{
      "Classification": "spark-defaults",
      "Properties": {
        "spark.executor.memory": "18g",
        "spark.executor.cores": "5",
        "spark.driver.memory": "8g",
        "spark.driver.cores": "4",
        "spark.executor.memoryOverhead": "2g",
        "spark.sql.shuffle.partitions": "200",
        "spark.sql.adaptive.enabled": "true",
        "spark.sql.adaptive.coalescePartitions.enabled": "true",
        "spark.sql.adaptive.skewJoin.enabled": "true",
        "spark.hadoop.fs.s3a.committer.name": "magic",
        "spark.hadoop.fs.s3a.fast.upload": "true",
        "spark.hadoop.fs.s3a.multipart.size": "134217728"
      }
    },
    {
      "Classification": "spark",
      "Properties": {
        "maximizeResourceAllocation": "true"   ← auto-size executors for the instance type
      }
    }]'

maximizeResourceAllocation:
  EMR automatically computes executor.memory and executor.cores
  based on the EC2 instance type
  ✓ Great starting point — but often leaves too much overhead
  ✓ Override individually after benchmarking

S3 performance configs:
  fs.s3a.committer.name=magic   → atomic renames using PUT instead of COPY+DELETE
  fs.s3a.fast.upload=true       → stream directly to S3 (no temp local disk)
  fs.s3a.multipart.size=128MB   → multipart upload threshold
  fs.s3a.threads.max=64         → concurrent S3 connections
══════════════════════════════════════════════════════════════
""")

## Part 4: EMR Serverless

In [ ]:
print("""
EMR Serverless — Patterns:
══════════════════════════════════════════════════════════════

EMR Serverless: no cluster management, pay per second of compute used.

# 1. Create an application (one-time setup)
aws emr-serverless create-application \\
  --name "my-spark-app" \\
  --type SPARK \\
  --release-label emr-6.15.0 \\
  --initial-capacity '{
    "DRIVER": {"workerCount": 1, "workerConfiguration": {"cpu": "4vCPU", "memory": "16GB"}},
    "EXECUTOR": {"workerCount": 10, "workerConfiguration": {"cpu": "4vCPU", "memory": "16GB"}}
  }'

# 2. Submit a job run
aws emr-serverless start-job-run \\
  --application-id app-xxxx \\
  --execution-role-arn arn:aws:iam::123456789:role/EMRServerlessRole \\
  --job-driver '{
    "sparkSubmit": {
      "entryPoint": "s3://my-bucket/scripts/etl_job.py",
      "entryPointArguments": ["--date", "2024-01-15"],
      "sparkSubmitParameters": "--conf spark.executor.cores=4 --conf spark.executor.memory=14g"
    }
  }' \\
  --configuration-overrides '{
    "monitoringConfiguration": {
      "s3MonitoringConfiguration": {"logUri": "s3://my-logs/emr-serverless/"}
    }
  }'

# 3. Check status
aws emr-serverless get-job-run --application-id app-xxxx --job-run-id run-yyyy

EMR Serverless vs EMR EC2:
  Feature             EMR Serverless              EMR EC2
  ─────────────────── ──────────────────────────  ─────────────────────
  Cluster management  None                        Full (sizing, tuning)
  Startup time        30-60 sec (pre-init: 0s)    5-15 min
  Cost model          Per vCPU-sec + GB-sec        Per node-hour
  Idle cost           $0                           EC2 rate × idle time
  Spot support        Built-in                     Manual instance groups
  Custom packages     Via S3 archive               Bootstrap action
  Streaming           Not supported                Yes
  Best for            Sporadic batch jobs          High-frequency/streaming

Pre-initialized capacity:
  Pre-warm workers to eliminate cold start for time-sensitive jobs
  Cost: you pay for pre-initialized workers even when idle
  Use when: SLA requires job to start within seconds
══════════════════════════════════════════════════════════════
""")

## Part 5: Step Functions + EMR Integration

In [ ]:
print("""
AWS Step Functions + EMR — Orchestration Pattern:
══════════════════════════════════════════════════════════════

Step Functions is AWS's serverless workflow orchestrator.
Use it to: create EMR cluster → submit steps → wait → terminate cluster.

State machine definition (simplified JSON):
{
  "Comment": "Daily ETL Pipeline",
  "StartAt": "CreateEMRCluster",
  "States": {
    "CreateEMRCluster": {
      "Type": "Task",
      "Resource": "arn:aws:states:::elasticmapreduce:createCluster.sync",
      "Parameters": {
        "Name": "ETL Cluster",
        "ReleaseLabel": "emr-6.15.0",
        "Applications": [{"Name": "Spark"}],
        "Instances": {
          "MasterInstanceType": "m5.xlarge",
          "SlaveInstanceType": "m5.2xlarge",
          "InstanceCount": 5
        }
      },
      "ResultPath": "$.cluster",
      "Next": "RunBronzeStep"
    },
    "RunBronzeStep": {
      "Type": "Task",
      "Resource": "arn:aws:states:::elasticmapreduce:addStep.sync",
      "Parameters": {
        "ClusterId.$": "$.cluster.ClusterId",
        "Step": {
          "Name": "Bronze Ingest",
          "ActionOnFailure": "TERMINATE_CLUSTER",
          "HadoopJarStep": {
            "Jar": "command-runner.jar",
            "Args": ["spark-submit", "--deploy-mode", "cluster",
                     "s3://my-bucket/scripts/bronze_ingest.py"]
          }
        }
      },
      "Next": "RunSilverStep"
    },
    "RunSilverStep": { "...": "same pattern" },
    "TerminateCluster": {
      "Type": "Task",
      "Resource": "arn:aws:states:::elasticmapreduce:terminateCluster.sync",
      "Parameters": {"ClusterId.$": "$.cluster.ClusterId"},
      "End": true
    }
  }
}

Step Functions vs Airflow for EMR:
  Step Functions:  AWS-native, no server to manage, event-driven triggers,
                   built-in retry, visual graph in AWS Console
  Airflow:         More flexible DAGs, richer operators ecosystem, better
                   cross-platform (not AWS-only), sensor-based dependencies
  Choose Step Functions: AWS-only shop, want zero infra overhead
  Choose Airflow: multi-cloud, complex cross-system dependencies
══════════════════════════════════════════════════════════════
""")

## Exercises

1. Design an EMR cluster for a job that processes 2 TB of S3 data daily. Recommend: instance type, instance count per group, Spot vs On-Demand allocation, and key Spark configurations. Justify each choice.
2. Write a bootstrap.sh script that installs a private Python package from a private PyPI server, copies a configuration file from S3, and only runs specific steps on the master node.
3. Compare the total cost of running a 4-hour daily ETL job on: (a) EMR EC2 with 10 m5.2xlarge nodes, (b) EMR Serverless with equivalent compute. Use AWS pricing.
4. What happens to in-flight Spark tasks when a Spot Task node is reclaimed by AWS? How does EMR handle this automatically? What configuration prevents excessive task loss?
5. Design a Step Functions state machine that: creates an EMR cluster, runs three sequential Spark steps, sends an SNS notification on success, terminates the cluster on both success and failure.